# ML Tradicional — UCI 350 + FLAML (Google Colab)

Roda **no navegador**, sem instalar Python no PC e sem Docker.

- Dataset: [UCI 350 — Default of Credit Card Clients](https://archive-beta.ics.uci.edu/dataset/350/default+of+credit+card+clients)
- Ferramenta: **FLAML AutoML**
- Alvo: `default_payment_next_month`

**Como usar:** *Runtime* → *Run all*

In [ ]:
!pip install -q flaml pandas openpyxl xlrd==1.2.0 scikit-learn joblib

In [ ]:
import io
import zipfile
from pathlib import Path

import pandas as pd
import requests

TARGET = "default_payment_next_month"
UCI_ZIP = "https://cdn.uci-ics-mlr-prod.aws.uci.edu/350/default+of+credit+card+clients.zip"
BETA_RENAME = {
    "X1": "LIMIT_BAL", "X2": "SEX", "X3": "EDUCATION", "X4": "MARRIAGE", "X5": "AGE",
    "X6": "PAY_0", "X7": "PAY_2", "X8": "PAY_3", "X9": "PAY_4", "X10": "PAY_5", "X11": "PAY_6",
    "X12": "BILL_AMT1", "X13": "BILL_AMT2", "X14": "BILL_AMT3", "X15": "BILL_AMT4",
    "X16": "BILL_AMT5", "X17": "BILL_AMT6", "X18": "PAY_AMT1", "X19": "PAY_AMT2",
    "X20": "PAY_AMT3", "X21": "PAY_AMT4", "X22": "PAY_AMT5", "X23": "PAY_AMT6", "Y": TARGET,
}

zip_bytes = requests.get(UCI_ZIP, timeout=120).content
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    xls_name = next(n for n in zf.namelist() if n.lower().endswith(".xls"))
    raw = pd.read_excel(io.BytesIO(zf.read(xls_name)), header=1)

df = raw.rename(columns={k: v for k, v in BETA_RENAME.items() if k in raw.columns})
if "default payment next month" in df.columns:
    df = df.rename(columns={"default payment next month": TARGET})
df.columns = [str(c).strip() for c in df.columns]
print(f"Linhas: {len(df):,} | Colunas: {len(df.columns)}")
df.head()

In [ ]:
print("Inadimplentes (%):", round(100 * df[TARGET].mean(), 2))
df[TARGET].value_counts().plot(kind="bar", title="Distribuição do alvo")
df.describe().T.head(12)

In [ ]:
DROP = ["ID"]  # cientista de dados: remover identificador
feature_cols = [c for c in df.columns if c not in DROP and c != TARGET]
X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(df[feature_cols].median(numeric_only=True))
y = df[TARGET].astype(int)
print("Features:", len(feature_cols))

In [ ]:
from flaml import AutoML
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

automl = AutoML()
automl.fit(
    X_train=X_train,
    y_train=y_train,
    task="classification",
    metric="roc_auc",
    time_budget=120,
    estimator_list=["rf", "extra_tree", "lrl1"],
    eval_method="cv",
    n_splits=5,
    seed=42,
    ensemble=True,
)

y_pred = automl.predict(X_test)
y_proba = automl.predict_proba(X_test)[:, 1]
print("Melhor estimador:", automl.best_estimator)
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("F1:", round(f1_score(y_test, y_pred), 4))
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nMatriz de confusão:\n", confusion_matrix(y_test, y_pred))
print("\n", classification_report(y_test, y_pred))

In [ ]:
import joblib
from google.colab import files

bundle = {
    "automl": automl,
    "feature_columns": feature_cols,
    "target": TARGET,
    "dropped": DROP,
}
model_path = "credit_default_flaml.pkl"
joblib.dump(bundle, model_path)
files.download(model_path)
print("Modelo baixado para o PC (.pkl)")